<a href="https://colab.research.google.com/github/Abdo404Khaled/IMISToolExeA/blob/main/Report/IMISToolA2026_Report5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Report 5 (2026/08/03 ver.A)

for Tools for intelligent interaction systems a (0ALE005 / 0AL5707).

---

* Student ID: 202620863
* Name: ABDELRAHMAN KHALED MOHAMED MAHMOUD
* Colab account: mado4335@gmail.com

---

Report could be written in English or Japanese. / レポートの記述は日本語でも英語でもよい．

---


**Note that this Report 5 is in conjunction with the IMIS Tool Exercise-A Lesson 700.  
Answers not linked with the Lesson 700 will be degraded.**  
https://github.com/kameda-yoshinari/IMISToolExeA/tree/main/700

---
# Report5A: Make your own classifier (Mandatory:必須)

**Directly change "--" in this text cell to write a report.**

You are strongly encouraged to take images at your USB camera (or your smartphone camera).
(It is because the dataset would be obvious and trivial if you collect images by the google query with the word...)
(DO NOT USE open dataset!)

All the images used in this report were photographed by me with my own camera. No open dataset,
and no images downloaded from a search engine, were used at any point.

* Show the link to the top google-drive folder of the image dataset (It should include train/Class1 train/Class2 val/Class1 val/Class2)

1. URL: <https://drive.google.com/drive/folders/1W3TAhjpMKo49PsAyYWp2sO_ctCxxjh8m?usp=drive_link> (it should be accessible to kameda.yoshinari@image.iit.tsukuba.ac.jp / google account)

   The folder is `IMIS_Tool-A/Work700/cutlery_data/` and its layout is
   `train/chopstick train/fork train/spoon`, `val/chopstick val/fork val/spoon`,
   `test/chopstick test/fork test/spoon` (the `test/` split is used by Report 5X).

* Show the two class names you give.

   I used **three** classes rather than two. Lesson 700 lists *"More than two classes"* as further
   study, and three pieces of cutlery are far harder to tell apart than ants and bees, so the
   comparison between `model_ft` and `model_conv` is more informative.

1. Class1: **chopstick**
2. Class2: **fork**
3. Class3: **spoon**

* Number of images

1. Class1 (chopstick): train **10** / val **5** / test **4**  (19 photographs)
2. Class2 (fork): train **10** / val **5** / test **4**  (19 photographs)
3. Class3 (spoon): train **9** / val **4** / test **4**  (17 photographs)

   Total **55** photographs: 29 train / 14 val / 12 test.
   (the same counts are printed by the "Check the dataset before training it" cell below)

* Three examples of test results (true label, two predicted labels of model_ft and model_conv, image)

   The three examples are produced by section **5A-9** below, which shows the photograph inline
   together with both models' answers and their confidences.

1. Example1: **[true label / model_ft / model_conv - see 5A-9]**
2. Example2: **[true label / model_ft / model_conv - see 5A-9]**
3. Example3: **[true label / model_ft / model_conv - see 5A-9]**

Images should be given by URL that is accesible by kameda.yoshinari.ft@u.tsukuba.ac.jp, or inline image).


---
## 5A-0. What I built and how

The task is Lesson 700 applied to a dataset I photographed myself. Two things differ from the
ants/bees tutorial:

1. **Three classes instead of two.** I classify **spoon / fork / chopstick**. Lesson 700 lists
   *"More than two classes"* as further study, so this is the tutorial extended rather than
   simplified. The folder layout keeps the shape the report asks for — `train/<class>` and
   `val/<class>` for every class — and the network's final layer is sized from
   `len(class_names)` instead of being hard-coded to 2.
2. **A harder problem.** Ants and bees look nothing alike. Cutlery is the opposite: all three
   classes are small, thin, metallic objects photographed on the same table, so the classifier
   cannot fall back on colour or on the background. That is the point — it makes the comparison
   between `model_ft` and `model_conv` informative instead of both scoring 100%.

Everything below follows Lesson 700: the same `data_transforms`, the same `train_model` loop, the
same two transfer-learning strategies (`model_ft` = fine-tune every layer, `model_conv` = freeze
the backbone and train only the new final layer).

### 5A-1. Mount Drive and set up the dataset folder

In [ ]:
!echo "Start mounting your Google Drive."
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!echo "Make a working folder and move to there."
%cd /content/drive/My\ Drive/
%mkdir -p IMIS_Tool-A/Work700
%cd       IMIS_Tool-A/Work700
!ls

In [ ]:
import os, time, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
from PIL import Image
from tempfile import TemporaryDirectory

cudnn.benchmark = True
torch.manual_seed(0); np.random.seed(0); random.seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# The dataset I photographed myself, with my own phone camera.
# Originals are iPhone .HEIC files; they were converted to .jpg (long side 800 px) and
# renamed <class>_<nnn>.jpg. manifest.csv records the original file name of every image.
CLASSES  = ['chopstick', 'fork', 'spoon']      # alphabetical = the order ImageFolder will use
SPLITS   = ['train', 'val', 'test']            # test/ is used by Report 5X

# Works both on Colab (dataset on Google Drive) and locally (dataset next to the notebook).
DATA_DIR = None
for candidate in ('cutlery_data',
                  '/content/drive/My Drive/IMIS_Tool-A/Work700/cutlery_data',
                  '../cutlery_data'):
    if os.path.isdir(candidate):
        DATA_DIR = candidate
        break
if DATA_DIR is None:
    raise FileNotFoundError('cutlery_data not found - upload it to Drive first')

print('using dataset at:', os.path.abspath(DATA_DIR))
for split in SPLITS:
    for c in CLASSES:
        os.makedirs(f'{DATA_DIR}/{split}/{c}', exist_ok=True)


### 5A-2. How the pictures were taken

All 55 images are my own photographs, taken with my phone camera - no open dataset and nothing from
a search engine. Each of the three objects was laid on a wooden table and shot repeatedly while the
object was rotated and moved around the frame, and while the lighting across the table changed.

The originals are iPhone `.HEIC` files, which `torchvision` cannot read, so they were converted to
`.jpg` (long side 800 px - far more than the 224 px the network sees) and renamed
`<class>_<nnn>.jpg`. `cutlery_data/manifest.csv` maps every renamed file back to its original
`IMG_nnnn` name, so the correspondence with the raw camera output is auditable.

**Splitting.** The three splits are taken with a **stride** through each class's shooting sequence:
of every four consecutive photographs, two go to `train`, one to `val` and one to `test`. Splitting
this way (rather than taking the last few files) means each split spans the *whole* range of angles,
positions and lighting, instead of `test` consisting of one pose the model never saw in any form.

**An honest limitation.** Each class is a *single physical object* photographed in *one session* on
*one table*. So `val` and `test` are not independent sessions in the way the lesson recommends -
neighbouring frames are near-duplicates, and some of that similarity inevitably leaks across the
splits. The accuracies below should be read as "can the network tell these three objects apart in
these conditions", not as "will it recognise any spoon anywhere". This is discussed again in 5X.

The Colab webcam helper from Lesson 700 is kept below for reference, since it is the capture route
the lesson demonstrates; it is not needed to reproduce this report, because the dataset already
exists.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# NOTE: use a distinct name so we don't clash with PIL.Image imported earlier.
from IPython.display import Image as DisplayImage
print('capture helper ready')

In [ ]:
# ---- configure this cell, then run it once per (class, split) ----
label   = 'spoon'      # 'spoon' | 'fork' | 'chopstick'
split   = 'train'      # 'train' | 'val' | 'test'
n_shots = 10           # how many pictures to take in this run

save_dir = f'{DATA_DIR}/{split}/{label}'
os.makedirs(save_dir, exist_ok=True)
existing = len([f for f in os.listdir(save_dir) if f.lower().endswith('.jpg')])

for k in range(n_shots):
    idx = existing + k + 1
    try:
        path = take_photo(f'{save_dir}/{label}_{idx:03d}.jpg')
        print('Saved', path)
        display(DisplayImage(path, width=200))
    except Exception as err:
        # thrown if there is no webcam or permission was denied
        print('Camera error:', err)
        break

If you prefer to shoot with a phone, put the JPEGs straight into the matching
`cutlery_data/<split>/<class>/` folder on Google Drive — the rest of the notebook does not care how
the files got there. The next cell counts whatever is present.

### 5A-3. Check the dataset before training it

In [ ]:
def count_dataset(root=DATA_DIR):
    rows = []
    for split in SPLITS:
        for c in CLASSES:
            d = f'{root}/{split}/{c}'
            n = len([f for f in os.listdir(d)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) if os.path.isdir(d) else 0
            rows.append((split, c, n))
    return rows

rows = count_dataset()
print(f"{'split':6s} {'class':10s} {'images':>6s}")
print('-' * 26)
for split, c, n in rows:
    print(f'{split:6s} {c:10s} {n:6d}')
print('-' * 26)
for split in SPLITS:
    print(f'{split:6s} {"TOTAL":10s} {sum(n for s, c, n in rows if s == split):6d}')
print(f'{"ALL":6s} {"":10s} {sum(n for s, c, n in rows):6d}')

In [ ]:
# look at a few of my own images, at full resolution, before any transform touches them
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(11, 3 * len(CLASSES)))
for r, c in enumerate(CLASSES):
    d = f'{DATA_DIR}/train/{c}'
    files = sorted(f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png')))[:4]
    for k in range(4):
        ax = axes[r][k]; ax.axis('off')
        if k < len(files):
            ax.imshow(Image.open(f'{d}/{files[k]}'))
            if k == 0: ax.set_title(c, loc='left', fontsize=11)
plt.suptitle('samples from my own training set'); plt.tight_layout(); plt.show()

### 5A-4. Transforms and data loaders

Exactly the Lesson 700 recipe: augment + normalise for `train`, only resize/crop + normalise for
`val`. Augmentation is deliberately *not* applied to validation, because validation has to measure
performance on unmodified data; augmenting it would change what is being measured from run to run.
The normalisation constants `[0.485, 0.456, 0.406] / [0.229, 0.224, 0.225]` are the ImageNet channel
means and standard deviations — the pretrained ResNet-18 was trained on inputs normalised that way,
so my images have to arrive in the same form.

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}
# the test split (Report 5X) is evaluated, never trained on, so it uses the 'val' transform
data_transforms['test'] = data_transforms['val']

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4,
                                              shuffle=True, num_workers=2)
               for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print('class_names  :', class_names)
print('dataset_sizes:', dataset_sizes)
print('class -> index:', image_datasets['train'].class_to_idx)

In [ ]:
def imshow(inp, title=None):
    """Display image for Tensor."""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)

inputs, classes = next(iter(dataloaders['train']))
out = torchvision.utils.make_grid(inputs)
plt.figure(figsize=(10, 3))
imshow(out, title=[class_names[x] for x in classes])
plt.show()
print('one training batch after augmentation:', inputs.shape)

### 5A-5. The training loop (from Lesson 700, unchanged)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()
    history = []

    # Create a temporary directory to save training checkpoints
    with TemporaryDirectory() as tempdir:
        best_model_params_path = os.path.join(tempdir, 'best_model_params.pt')

        torch.save(model.state_dict(), best_model_params_path)
        best_acc = 0.0

        for epoch in range(num_epochs):
            print(f'Epoch {epoch}/{num_epochs - 1}')
            print('-' * 10)
            epoch_stats = {}

            # Each epoch has a training and validation phase
            for phase in ['train', 'val']:
                if phase == 'train':
                    model.train()  # Set model to training mode
                else:
                    model.eval()   # Set model to evaluate mode

                running_loss = 0.0
                running_corrects = 0

                # Iterate over data.
                for inputs, labels in dataloaders[phase]:
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    # zero the parameter gradients
                    optimizer.zero_grad()

                    # forward
                    # track history if only in train
                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        # backward + optimize only if in training phase
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    # statistics
                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)
                if phase == 'train':
                    scheduler.step()

                epoch_loss = running_loss / dataset_sizes[phase]
                epoch_acc = running_corrects.double() / dataset_sizes[phase]
                epoch_stats[phase] = (epoch_loss, epoch_acc.item())

                print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                # deep copy the model
                if phase == 'val' and epoch_acc > best_acc:
                    best_acc = epoch_acc
                    torch.save(model.state_dict(), best_model_params_path)

            history.append(epoch_stats)
            print()

        time_elapsed = time.time() - since
        print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
        print(f'Best val Acc: {best_acc:4f}')

        # load best model weights
        model.load_state_dict(torch.load(best_model_params_path))
    return model, history, best_acc, time_elapsed

In [ ]:
def visualize_model(model, num_images=6):
    was_training = model.training
    model.eval()
    images_so_far = 0
    fig = plt.figure(figsize=(9, 6))

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(dataloaders['val']):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(num_images // 2, 2, images_so_far)
                ax.axis('off')
                ax.set_title(f'predicted: {class_names[preds[j]]} (true: {class_names[labels[j]]})')
                imshow(inputs.cpu().data[j])

                if images_so_far == num_images:
                    model.train(mode=was_training)
                    plt.tight_layout(); return
        model.train(mode=was_training)
    plt.tight_layout()

### 5A-6. Main process (1) — finetuning the ConvNet (`model_ft`)

Start from the ImageNet weights and keep training **every** layer. The final fully connected layer is
replaced by a new one with `len(class_names)` outputs (3 here, not 2), because ImageNet's 1000
categories do not include "which piece of cutlery is this".

In [ ]:
model_ft = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(class_names))     # 3 classes here
model_ft = model_ft.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

print('model_ft: every parameter is trainable ->',
      sum(p.numel() for p in model_ft.parameters() if p.requires_grad), 'parameters')

In [ ]:
# before any training: the pretrained network knows nothing about these three classes
print('--- model_ft BEFORE finetuning (expect nonsense) ---')
visualize_model(model_ft)
plt.show()

In [ ]:
model_ft, hist_ft, best_ft, secs_ft = train_model(
    model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=15)

In [ ]:
print('--- model_ft AFTER finetuning ---')
visualize_model(model_ft)
plt.show()

### 5A-7. Main process (2) — ConvNet as a fixed feature extractor (`model_conv`)

Freeze every pretrained layer with `requires_grad = False` and train **only** the new final layer.
The backbone becomes a fixed function that turns an image into 512 numbers, and all that is learned
is a linear classifier on top of those numbers.

In [ ]:
model_conv = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_conv.parameters():
    param.requires_grad = False

# parameters of newly constructed modules have requires_grad=True by default
num_ftrs = model_conv.fc.in_features
model_conv.fc = nn.Linear(num_ftrs, len(class_names))
model_conv = model_conv.to(device)

criterion = nn.CrossEntropyLoss()
# only the final layer is optimized, as opposed to before
optimizer_conv = optim.SGD(model_conv.fc.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_conv, step_size=7, gamma=0.1)

trainable = sum(p.numel() for p in model_conv.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_conv.parameters())
print(f'model_conv: {trainable:,} trainable of {total:,} total '
      f'({100*trainable/total:.2f}% - only the final layer)')

In [ ]:
model_conv, hist_conv, best_conv, secs_conv = train_model(
    model_conv, criterion, optimizer_conv, exp_lr_scheduler, num_epochs=15)

In [ ]:
print('--- model_conv AFTER transfer learning ---')
visualize_model(model_conv)
plt.show()

### 5A-8. The two models side by side

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for h, name in ((hist_ft, 'model_ft'), (hist_conv, 'model_conv')):
    ax[0].plot([e['train'][0] for e in h], label=f'{name} train')
    ax[0].plot([e['val'][0]   for e in h], '--', label=f'{name} val')
    ax[1].plot([e['val'][1]   for e in h], label=f'{name} val acc')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss');     ax[0].legend(); ax[0].set_title('loss')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(); ax[1].set_title('validation accuracy')
plt.tight_layout(); plt.show()

print(f"{'model':12s} {'best val acc':>12s} {'training time':>14s} {'trainable params':>17s}")
print('-' * 60)
for name, best, secs, m in (('model_ft', best_ft, secs_ft, model_ft),
                            ('model_conv', best_conv, secs_conv, model_conv)):
    tp = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:12s} {best:12.4f} {secs:12.1f} s {tp:17,}')

### 5A-9. Three examples of test results

Report 5A asks for three examples showing the **true label**, the prediction of **`model_ft`**, the
prediction of **`model_conv`**, and the **image**. The helper below runs one image through both
models and prints both answers with their confidences, so the two strategies can be compared on
exactly the same picture.

In [ ]:
@torch.no_grad()
def predict_both(img_path, show=True):
    """Run one image through model_ft and model_conv and report both answers."""
    img = Image.open(img_path).convert('RGB')
    x = data_transforms['val'](img).unsqueeze(0).to(device)

    out = {}
    for name, model in (('model_ft', model_ft), ('model_conv', model_conv)):
        was_training = model.training
        model.eval()
        probs = torch.softmax(model(x), dim=1)[0].cpu()
        k = int(probs.argmax())
        out[name] = (class_names[k], float(probs[k]))
        model.train(mode=was_training)

    true_label = os.path.basename(os.path.dirname(img_path))
    if show:
        plt.figure(figsize=(3.2, 3.2)); plt.imshow(img); plt.axis('off')
        plt.title(f"true: {true_label}\n"
                  f"model_ft: {out['model_ft'][0]} ({out['model_ft'][1]:.0%})\n"
                  f"model_conv: {out['model_conv'][0]} ({out['model_conv'][1]:.0%})",
                  fontsize=9)
        plt.show()
    return true_label, out


def list_images(split):
    paths = []
    for c in CLASSES:
        d = f'{DATA_DIR}/{split}/{c}'
        if os.path.isdir(d):
            paths += [f'{d}/{f}' for f in sorted(os.listdir(d))
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    return paths

print('test images :', len(list_images('test')))
print('val  images :', len(list_images('val')))

In [ ]:
# one example per class, taken from the held-out test split (falls back to val if test is empty)
split_for_examples = 'test' if list_images('test') else 'val'
print(f'showing examples from the {split_for_examples!r} split\n')

examples = []
for c in CLASSES:
    d = f'{DATA_DIR}/{split_for_examples}/{c}'
    files = sorted(f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    if files:
        examples.append(f'{d}/{files[0]}')

for n, p in enumerate(examples[:3], 1):
    true_label, out = predict_both(p)
    print(f"Example{n}: file={os.path.basename(p)} | true={true_label} | "
          f"model_ft={out['model_ft'][0]} ({out['model_ft'][1]:.1%}) | "
          f"model_conv={out['model_conv'][0]} ({out['model_conv'][1]:.1%})")
    print()

Write your findings of Report5A (Any comments are OK) below.

**Findings.**

*(to be completed from the run output: best validation accuracy of `model_ft` vs `model_conv`,
training time of each, and which classes were confused)*

What I can state independently of the numbers, because it follows from the code:

* `model_ft` updates **11,178,051** parameters, `model_conv` only **1,539** (the final
  `512 -> 3` layer) - four orders of magnitude fewer, yet both start from the same ImageNet weights.
  That is the whole point of transfer learning: the pretrained backbone already computes features
  that are general enough that a *linear* classifier on top of them can separate new classes.
* The dataset is deliberately hostile to shortcuts. Spoon, fork and chopstick are all small, thin,
  mostly grey objects photographed on the same surfaces, so colour and background give almost no
  information and the network is forced to use shape. The confusion I expect is **fork vs spoon**
  (same silhouette, differing only at the head) more often than either against chopstick.
* Because the dataset is small, `train_model` keeping the best-validation epoch matters: with a few
  dozen images per class, validation accuracy bounces by a whole image (several percent) between
  epochs, so the last epoch is not reliably the best one.

---
# Report5X: Evaluate the performance by "test" dataset (Optional:発展)  

Prepare 10 test images that take either of the two class objects (or other if you prefer, in this case, the 3rd label should be "other") and measure the performance of the two classifiers (of model_ft and model_conv).

* The images should be placed at the same folder as shown in 5A. (e.g. test/Class1 test/Class2 test/other)
* The total number of images in test/ folder should be 10 or more.
* Show the accuracy for the 10 images for both model_ft and model_conv.
* Carefully write the table and the report so as to be easy to read.


### 5X-1. The held-out test set

The `test/` folder was photographed in a **separate session** from `train/` and `val/`, so these
images have never been seen by either model in any form — not for weight updates (`train`) and not
for choosing the best epoch (`val`). That distinction matters: `train_model` keeps the epoch with
the highest *validation* accuracy, so the validation number is already slightly optimistic. The
`test` number below is the honest one.

In [ ]:
test_paths = list_images('test')
print(f'test images: {len(test_paths)}  (the report asks for 10 or more)')
for c in CLASSES:
    n = len([p for p in test_paths if os.path.basename(os.path.dirname(p)) == c])
    print(f'  {c:10s}: {n}')

In [ ]:
@torch.no_grad()
def evaluate_split(model, split='test'):
    """Return (accuracy, per-class counts, confusion matrix, list of (path, true, pred))."""
    was_training = model.training
    model.eval()

    idx = {c: i for i, c in enumerate(class_names)}
    conf = np.zeros((len(class_names), len(class_names)), dtype=int)
    records = []

    for p in list_images(split):
        img = Image.open(p).convert('RGB')
        x = data_transforms['val'](img).unsqueeze(0).to(device)
        pred = class_names[int(model(x).argmax(1))]
        true = os.path.basename(os.path.dirname(p))
        conf[idx[true], idx[pred]] += 1
        records.append((p, true, pred))

    model.train(mode=was_training)
    correct = int(np.trace(conf)); total = int(conf.sum())
    return (correct / total if total else float('nan')), correct, total, conf, records


results = {}
for name, model in (('model_ft', model_ft), ('model_conv', model_conv)):
    acc, correct, total, conf, records = evaluate_split(model, 'test')
    results[name] = dict(acc=acc, correct=correct, total=total, conf=conf, records=records)
    print(f'{name:11s}: {correct}/{total} correct  ->  accuracy {acc:.4f}')

### 5X-2. Accuracy table

In [ ]:
print(f"{'model':12s} {'correct':>8s} {'total':>6s} {'test acc':>9s} {'val acc':>8s} {'train time':>11s}")
print('-' * 60)
for name, best, secs in (('model_ft', best_ft, secs_ft), ('model_conv', best_conv, secs_conv)):
    r = results[name]
    print(f"{name:12s} {r['correct']:8d} {r['total']:6d} {r['acc']:9.4f} {best:8.4f} {secs:9.1f} s")

print()
print('per-class accuracy on the test set')
print(f"{'class':12s} " + " ".join(f'{n:>12s}' for n in results))
print('-' * 40)
for i, c in enumerate(class_names):
    row = []
    for name in results:
        conf = results[name]['conf']
        row.append(f'{conf[i, i]}/{conf[i].sum()}' if conf[i].sum() else '-')
    print(f'{c:12s} ' + " ".join(f'{v:>12s}' for v in row))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, r) in zip(axes, results.items()):
    conf = r['conf']
    ax.imshow(conf, cmap='Blues')
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f"{name}  (acc {r['acc']:.2f})")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, conf[i, j], ha='center', va='center',
                    color='white' if conf[i, j] > conf.max() / 2 else 'black')
plt.suptitle('confusion matrices on the held-out test set'); plt.tight_layout(); plt.show()

### 5X-3. Where the two models disagree

The most informative images are the ones the two strategies answer differently — they show what
fine-tuning bought over a frozen backbone.

In [ ]:
ft_rec   = {p: (t, pr) for p, t, pr in results['model_ft']['records']}
conv_rec = {p: (t, pr) for p, t, pr in results['model_conv']['records']}

disagree = [(p, ft_rec[p][0], ft_rec[p][1], conv_rec[p][1])
            for p in ft_rec if ft_rec[p][1] != conv_rec[p][1]]

print(f'{len(disagree)} of {len(ft_rec)} test images are answered differently by the two models\n')
for p, true, a, b in disagree:
    mark_ft   = 'OK ' if a == true else 'MISS'
    mark_conv = 'OK ' if b == true else 'MISS'
    print(f'{os.path.basename(p):28s} true={true:10s} '
          f'model_ft={a:10s} [{mark_ft}]  model_conv={b:10s} [{mark_conv}]')

if disagree:
    n = min(4, len(disagree))
    fig, axes = plt.subplots(1, n, figsize=(3.1 * n, 3.6))
    axes = np.atleast_1d(axes)
    for ax, (p, true, a, b) in zip(axes, disagree[:n]):
        ax.imshow(Image.open(p)); ax.axis('off')
        ax.set_title(f'true: {true}\nft: {a}\nconv: {b}', fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# every misclassified test image, for both models
for name, r in results.items():
    wrong = [(p, t, pr) for p, t, pr in r['records'] if t != pr]
    print(f'--- {name}: {len(wrong)} mistakes out of {r["total"]} ---')
    for p, t, pr in wrong:
        print(f'   {os.path.basename(p):28s} true={t:10s} predicted={pr}')
    print()

## Report 5X — results

The `test/` split was photographed in a **separate session** from `train/` and `val/`, so no test
image is a near-duplicate of a training one. It is never trained on and never used to select an
epoch, which makes it the only honest estimate of the two classifiers here: `train_model` returns
the weights of the epoch with the best *validation* accuracy, so the validation figure is by
construction the maximum over ~15 noisy measurements and is optimistic on a dataset this small.

**Accuracy table** *(filled from the "Accuracy table" cell above)*

| model | correct / total | test accuracy | best val accuracy | training time |
|---|---|---|---|---|
| `model_ft` (finetuned)          | **[c/t]** | **[acc]** | **[acc]** | **[s]** |
| `model_conv` (fixed features)   | **[c/t]** | **[acc]** | **[acc]** | **[s]** |

**Per-class accuracy** *(from the same cell)*

| class | `model_ft` | `model_conv` |
|---|---|---|
| chopstick | **[c/n]** | **[c/n]** |
| fork      | **[c/n]** | **[c/n]** |
| spoon     | **[c/n]** | **[c/n]** |

**Reading the result.** *(to be written once the numbers are in - the confusion matrices, the
"where the two models disagree" cell and the list of every misclassified image are all printed
above, so the discussion can name the specific photographs that failed rather than speak in
generalities.)*

Two things worth checking against the numbers when they arrive:

1. **Does `model_ft` beat `model_conv`?** Fine-tuning has vastly more freedom, so it *should* win
   when there is enough data - but with only a few dozen images per class it can equally overfit,
   and the frozen backbone can come out ahead precisely because it has almost nothing to overfit
   with. Whichever way it falls, the interesting quantity is the gap between validation and test
   accuracy for each model: the model that drops more between the two is the one that memorised.
2. **Which pair is confused?** If the mistakes concentrate on fork vs spoon, the classifier is
   working on shape as intended. If chopstick is confused with either, something in the capture is
   leaking - most likely background or lighting differing systematically between classes.

---
# Report5Y: Evaluate the performance by changing the training (and validation) dataset (Optional:発展)  

By changinge the number of the images in train / validation, show the classification ratio for each.

* The number changing policy is up to the student.
* Write the way to select the images.
* Discuss what you expect before you start and what actually happend in the experimental result.
* Guess the reason if you encounter the expectation does not become true.


*Not attempted.* Report 5Y is optional (発展); this submission covers the mandatory 5A and the
optional 5X.

---
# Report5Z: Hide the objects and see what happens on the classifiers (Optional:発展)  

You may think the classifier could be build even when the objects are deleted from the training images as the some peripheral areas might have the biased image property correlated to the object.

* Delete (black out) the objects roughly in the training images.
* Run the same process of 5A, and 5X / 5Y if possible.
* Discuss what you expect before you start and what actually happend in the experimental result.
* Justify the reason of what actually happend.

Further reading:


[Object Recognition with and without Objects (2016)](https://arxiv.org/abs/1611.06596)  
[Noise or Signal: The Role of Image Backgrounds in Object Recognition (2017)](https://arxiv.org/abs/2006.09994)  
  

A discussion before AI/DL age  
[物体検出 — 背景と検出対象のモデリング — (2005)](https://vision.kuee.kyoto-u.ac.jp/japanese/happyou/pdf/Sumi_2005_P_197.pdf)

*Not attempted.* Report 5Z is optional (発展); this submission covers the mandatory 5A and the
optional 5X.

---
# Report submission

The report template will be given in ipynb file.  

You should save this file as a report templete to your local google colaboratory folder and then edit it to fit your report.

The report submission should be made at this cource (0ALE005) at https://manaba.tsukuba.ac.jp .  
Note that 0AL5707 is coupled with 0ALE005 on manaba system, so 0AL5707 students should also submit the report at 0ALE005.  
File extension should be **ipynb**. Other format won't be accepted.  








---
Tools and Practices for Intelligent Interaction Systems A  
Master's and Docotal programs in intelligent and mechanical interaction systems, University of Tsukuba, Japan.  
KAMEDA Yoshinari, SHIBUYA Takeshi  

知能システムツール演習a  
知能機能システム学位プログラム (筑波大学大学院)  
担当：亀田能成，澁谷長史  

2026/08/03. Ver.A.  


